# Data Acquisition — Sentinel-2 L2A and FIRMS

**Environment**: Local CPU

**Part 1**: Sentinel-2 L2A (6 scenes, 0% cloud cover) via CDSE STAC
**Part 2**: NASA FIRMS VIIRS active fire detections

## 1. Imports y credenciales

In [ ]:
import os
import time
import requests
import pystac_client
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(dotenv_path=Path("..") / ".env")

CDSE_USER     = os.getenv("CDSE_USER")
CDSE_PASSWORD = os.getenv("CDSE_PASSWORD")

DATA_DIR = Path("..") / "data" / "sentinel2" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Usuario CDSE: {CDSE_USER}")
print(f"Output dir:   {DATA_DIR.resolve()}")

## 2. AOI y ventana temporal

Bounding box over the most affected area of Corrientes (north and central province).  
Tres ventanas temporales: pre-fuego, pico, post-fuego.

In [ ]:
# Corrientes, Argentina — zona central/norte del incendio
BBOX = [-59.5, -29.0, -56.0, -26.5]  # [W, S, E, N]

# Tres ventanas: pre-fuego / pico de propagación / post-fuego
TIME_WINDOWS = [
    ("2021-12-01T00:00:00Z", "2021-12-31T23:59:59Z", "pre_fire"),
    ("2022-01-01T00:00:00Z", "2022-01-31T23:59:59Z", "peak_fire"),
    ("2022-02-01T00:00:00Z", "2022-02-28T23:59:59Z", "post_fire"),
]

# Bandas a descargar
BANDS = ["B02", "B03", "B04", "B08", "B8A", "B11", "B12", "SCL"]

MAX_CLOUD = 20  # % cobertura de nubes

print(f"BBOX: {BBOX}")
print(f"Ventanas temporales: {[w[2] for w in TIME_WINDOWS]}")
print(f"Bandas: {BANDS}")
print(f"Cloud cover máximo: {MAX_CLOUD}%")

## 3. Funciones auxiliares

In [ ]:
def get_cdse_token(user, password):
    """Obtiene token OAuth2 de CDSE. Válido ~10 minutos."""
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id": "cdse-public",
            "grant_type": "password",
            "username": user,
            "password": password,
        },
        timeout=30
    )
    r.raise_for_status()
    return r.json()["access_token"]


def download_file(href, token, output_path, chunk_size=1024*1024):
    """Descarga un asset de CDSE usando token Bearer."""
    headers = {"Authorization": f"Bearer {token}"}
    with requests.get(href, headers=headers, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(output_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True,
            desc=output_path.name, leave=False
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                bar.update(len(chunk))


print("Funciones definidas.")

## 4. Búsqueda en catálogo STAC

In [ ]:
# Nombre correcto de colección en CDSE: "sentinel-2-l2a" (minúsculas, con guiones)
# El parámetro query no está soportado — se filtra cloud cover en Python
catalog = pystac_client.Client.open(
    "https://catalogue.dataspace.copernicus.eu/stac"
)

all_items = []

for start, end, label in TIME_WINDOWS:
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=BBOX,
        datetime=f"{start}/{end}",
        max_items=100
    )
    items_raw = list(search.items())

    # Filtrar en Python: solo imágenes con cloud cover < MAX_CLOUD
    items = [
        it for it in items_raw
        if it.properties.get("eo:cloud_cover", 100) < MAX_CLOUD
    ]

    for item in items:
        item.extra_fields["window_label"] = label
    all_items.extend(items)
    print(f"  {label:12} → {len(items_raw):3} total, {len(items):3} con cloud<{MAX_CLOUD}%")

print(f"\nTotal seleccionados: {len(all_items)} items")

## 5. Inspeccionar assets del primer item

In [ ]:
if not all_items:
    raise RuntimeError("Sin resultados. Verificar BBOX, fechas o cloud cover.")

item = all_items[0]
print(f"Item: {item.id}")
print(f"Fecha: {item.datetime}")
print(f"Cloud: {item.properties.get('eo:cloud_cover')}%")
print(f"\nAssets disponibles:")
for key, asset in item.assets.items():
    print(f"  {key:<15} {asset.href[:80]}")

## 6. Selección de items por ventana

Para cada ventana temporal, seleccionamos las 2 imágenes con menor cloud cover.  
Esto da 6 fechas en total (2 pre, 2 peak, 2 post) — suficiente para construir pares T→T+1.

In [ ]:
from collections import defaultdict

by_window = defaultdict(list)
for item in all_items:
    by_window[item.extra_fields["window_label"]].append(item)

selected = []
for label in ["pre_fire", "peak_fire", "post_fire"]:
    items_sorted = sorted(
        by_window[label],
        key=lambda x: x.properties.get("eo:cloud_cover", 100)
    )
    top2 = items_sorted[:2]
    selected.extend(top2)
    for it in top2:
        date = it.datetime.strftime("%Y-%m-%d")
        cloud = it.properties.get("eo:cloud_cover")
        print(f"  {label:12} {date}  cloud={cloud:.1f}%  {it.id[:40]}")

print(f"\nItems seleccionados para descarga: {len(selected)}")

## 7. Descarga de bandas

In [ ]:
import re

# eodata.dataspace.copernicus.eu requiere S3 HMAC credentials, no Bearer token (→ 403).
# El endpoint correcto con Bearer OAuth2 es download.dataspace.copernicus.eu (OData API).
# UUID del producto: se extrae del asset 'Product' (zipper URL).
# Path de nodos: se construye desde el href S3 cortando desde .SAFE/ en adelante.

def get_product_uuid(item):
    href = item.assets["Product"].href
    m = re.search(r'Products\(([^)]+)\)', href)
    return m.group(1) if m else None

def s3_to_odata(s3_url, uuid):
    """Convierte s3://eodata/... a URL OData descargable con Bearer token."""
    path = s3_url.replace("s3://eodata/", "")
    parts = path.split("/")
    safe_idx = next(i for i, p in enumerate(parts) if p.endswith(".SAFE"))
    safe_name = parts[safe_idx]
    rest = parts[safe_idx + 1:]
    nodes = "/".join(f"Nodes({p})" for p in rest)
    return (
        f"https://download.dataspace.copernicus.eu"
        f"/odata/v1/Products({uuid})/Nodes({safe_name})/{nodes}/$value"
    )

BAND_ASSETS = {
    "B02": "B02_10m", "B03": "B03_10m", "B04": "B04_10m", "B08": "B08_10m",
    "B8A": "B8A_20m", "B11": "B11_20m", "B12": "B12_20m", "SCL": "SCL_20m",
}

token = get_cdse_token(CDSE_USER, CDSE_PASSWORD)
token_time = time.time()
skipped = []
downloaded = []

for item in selected:
    if time.time() - token_time > 540:
        token = get_cdse_token(CDSE_USER, CDSE_PASSWORD)
        token_time = time.time()
        print("Token refrescado.")

    uuid = get_product_uuid(item)
    date_str = item.datetime.strftime("%Y%m%d")
    tile_dir = DATA_DIR / item.id / date_str
    tile_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{item.id[:55]} — {date_str}")

    for band, asset_key in BAND_ASSETS.items():
        if asset_key not in item.assets:
            print(f"  SKIP  {band} (asset {asset_key} no encontrado)")
            skipped.append((item.id, band))
            continue

        output_path = tile_dir / f"{band}.jp2"
        if output_path.exists():
            print(f"  SKIP  {band} (ya existe)")
            continue

        odata_url = s3_to_odata(item.assets[asset_key].href, uuid)
        try:
            download_file(odata_url, token, output_path)
            size_mb = output_path.stat().st_size / 1e6
            print(f"  OK    {band} → {output_path.name}  ({size_mb:.1f} MB)")
            downloaded.append(output_path)
        except Exception as e:
            print(f"  ERROR {band}: {e}")
            if output_path.exists():
                output_path.unlink()
            skipped.append((item.id, band))

print(f"\nDescarga completa.")
print(f"  Archivos descargados: {len(downloaded)}")
print(f"  Skipped/errores:      {len(skipped)}")

## 8. Verificación rápida

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# Buscar B08.jp2 (formato nativo de S2)
jp2_files = list(DATA_DIR.rglob("B08.jp2"))

if jp2_files:
    path = jp2_files[0]
    with rasterio.open(path) as src:
        data = src.read(1)
        print(f"Archivo: {path}")
        print(f"CRS: {src.crs}")
        print(f"Shape: {data.shape}")
        print(f"Rango de valores: [{data.min()}, {data.max()}]")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(data, cmap="RdYlGn", vmin=0, vmax=3000)
    ax.set_title(f"B08 (NIR) — {path.parent.name}")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(Path("..") / "results" / "check_B08.png", dpi=100)
    plt.show()
    print("Imagen guardada en results/check_B08.png")

    print(f"\nResumen de archivos descargados:")
    all_jp2 = list(DATA_DIR.rglob("*.jp2"))
    print(f"  Total: {len(all_jp2)} archivos JP2")
    total_gb = sum(f.stat().st_size for f in all_jp2) / 1e9
    print(f"  Tamaño: {total_gb:.2f} GB")
else:
    print("No se encontraron archivos B08.jp2. Revisar la descarga.")

---

## Part 2 — NASA FIRMS VIIRS Active Fire

## 1. Imports y credenciales

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

FIRMS_KEY = os.getenv("FIRMS_API_KEY")

DATA_DIR = Path("..") / "data" / "firms"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"FIRMS key: {FIRMS_KEY[:8]}...")
print(f"Output dir: {DATA_DIR.resolve()}")

## 2. Parámetros de descarga

In [ ]:
# Corrientes, Argentina — mismo BBOX que Sentinel-2
BBOX_W, BBOX_S, BBOX_E, BBOX_N = -59.5, -29.0, -56.0, -26.5

# Período completo
START_DATE = datetime(2021, 12, 1)
END_DATE   = datetime(2022, 2, 28)

# Producto: VIIRS Suomi NPP Standard Product (histórico)
# NOTA: VIIRS_SNPP_SP tiene límite de days=1 por petición (no 10 como NRT)
PRODUCT = "VIIRS_SNPP_SP"
CHUNK_DAYS = 1

# Una petición por día
chunks = []
current = START_DATE
while current <= END_DATE:
    chunks.append((current, 1))
    current += timedelta(days=1)

print(f"Período: {START_DATE.date()} → {END_DATE.date()}")
print(f"Total peticiones: {len(chunks)} (1 día cada una)")
print(f"Tiempo estimado: ~{len(chunks) * 0.5 / 60:.1f} min")

## 3. Descarga por chunks

In [ ]:
def download_firms_day(key, product, bbox_w, bbox_s, bbox_e, bbox_n, date_str):
    bbox_str = f"{bbox_w},{bbox_s},{bbox_e},{bbox_n}"
    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{key}/{product}/{bbox_str}/1/{date_str}"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text


all_dfs = []
errors = []

for i, (start, _) in enumerate(chunks):
    date_str = start.strftime("%Y-%m-%d")
    output_path = DATA_DIR / f"viirs_{date_str}.csv"

    if output_path.exists():
        df = pd.read_csv(output_path)
        if len(df) > 0:
            all_dfs.append(df)
        continue

    try:
        csv_text = download_firms_day(
            FIRMS_KEY, PRODUCT, BBOX_W, BBOX_S, BBOX_E, BBOX_N, date_str
        )

        if csv_text.startswith("Error") or csv_text.startswith("<!DOCTYPE"):
            raise ValueError(f"API error: {csv_text[:100]}")

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(csv_text)

        lines = csv_text.strip().split("\n")
        n_det = len(lines) - 1
        if n_det > 0:
            from io import StringIO
            df = pd.read_csv(StringIO(csv_text))
            all_dfs.append(df)
            if i % 10 == 0 or n_det > 50:
                print(f"  {date_str}  {n_det:>5} detecciones")

        time.sleep(0.5)

    except Exception as e:
        print(f"  ERROR {date_str}: {e}")
        errors.append((date_str, str(e)))

print(f"\nDías procesados: {len(chunks) - len(errors)}/{len(chunks)}")
print(f"Errores: {len(errors)}")
if errors:
    print(errors[:3])

## 4. Consolidar en un único CSV

In [ ]:
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    df_all["acq_date"] = pd.to_datetime(df_all["acq_date"])
    df_all = df_all.sort_values("acq_date").reset_index(drop=True)

    output_path = DATA_DIR / "viirs_snpp_corrientes_2021-12_2022-02.csv"
    df_all.to_csv(output_path, index=False)

    print(f"CSV consolidado: {output_path}")
    print(f"Total detecciones: {len(df_all)}")
    print(f"Columnas: {list(df_all.columns)}")
    print(f"\nPrimeras filas:")
    print(df_all.head())
else:
    print("Sin datos para consolidar.")

## 5. Visualización — detecciones por día

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if 'df_all' in dir() and len(df_all) > 0:
    daily = df_all.groupby("acq_date").size().rename("detections")

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(daily.index, daily.values, alpha=0.6, color="#d62728")
    ax.plot(daily.index, daily.values, color="#d62728", linewidth=1)

    ax.axvspan(
        pd.Timestamp("2021-12-01"), pd.Timestamp("2021-12-31"),
        alpha=0.08, color="gray", label="pre-fire"
    )
    ax.axvspan(
        pd.Timestamp("2022-01-01"), pd.Timestamp("2022-01-31"),
        alpha=0.08, color="red", label="peak fire"
    )
    ax.axvspan(
        pd.Timestamp("2022-02-01"), pd.Timestamp("2022-02-28"),
        alpha=0.08, color="green", label="post-fire"
    )

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=45)
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Detecciones VIIRS")
    ax.set_title("Detecciones de fuego activo — Corrientes, Argentina (VIIRS SNPP)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    out = Path("..") / "results" / "firms_daily_detections.png"
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"Gráfico guardado en {out}")